# DeepSurv (PyCox) – Training Pipeline (Versione Aggiornata)

Questo notebook definisce l’addestramento di un modello **DeepSurv** basato su reti neurali, utilizzando la libreria PyCox.
Rispetto alla versione precedente, sono state introdotte diverse migliorie: **missing indicators**, un’architettura leggermente più ampia, una regolarizzazione più spinta (weight decay), una separazione esplicita di un **validation set** e un meccanismo di early stopping più paziente per sfruttare al meglio le epoche disponibili.

---

### Fase 1: Caricamento e Split Immediato

Il dataset viene caricato e i placeholder `-1` (utilizzati per codificare i valori mancanti) vengono convertiti in `NaN`.
Prima di qualsiasi trasformazione, il dataset viene suddiviso in **training (75%)** e **test (25%)** con stratificazione sull’evento (`vital_status_DEAD`). Questa suddivisione immediata è fondamentale per evitare qualsiasi forma di *data leakage*: tutte le successive operazioni (creazione di missing indicators, feature selection, imputazione, scaling) devono basarsi esclusivamente sulle statistiche del training set.

Contemporaneamente, per le variabili con alta percentuale di missing (es. `exposures.alcohol_days_per_week`, `exposures.alcohol_drinks_per_day`, `exposures.pack_years_smoked`), vengono create delle **missing indicators** binarie (colonna `_is_missing`) che segnalano se il valore è assente. Queste feature aggiuntive permettono alla rete di apprendere eventuali pattern legati alla mancanza stessa del dato.

---

### Fase 2: Feature Selection (Soglia di Frequenza) e Creazione del Validation Set

Dopo lo split, sul solo training set si identificano le **feature binarie con pochissimi casi positivi** (soglia minima `min_positives = 10`). Queste vengono rimosse perché potrebbero portare a instabilità o overfitting. La lista delle feature eliminate viene poi applicata anche al test set.

Successivamente, il training set viene ulteriormente diviso in **training (85% del training originale)** e **validation (15%)**, sempre stratificando sull’evento. Questa separazione è essenziale per monitorare l’andamento durante l’addestramento e implementare l’early stopping.

L’elenco delle feature finali (colonne rimaste dopo la selezione) viene salvato in un file `deepsurv_features.pkl` per garantire la coerenza in fase di valutazione.

---

### Fase 3: Imputazione MICE e Scaling

Le reti neurali non tollerano valori mancanti, pertanto si applica l’algoritmo **MICE** (`IterativeImputer`) per imputare i `NaN` residui. L’imputer viene addestrato **esclusivamente sul training set** (dopo la rimozione delle feature sparse) e poi applicato a training, validation e test. Il numero di iterazioni è fissato a `max_iter=10`, sufficiente per dataset di dimensioni moderate.

Successivamente, i dati imputati vengono **standardizzati** con `StandardScaler`, addestrato anch’esso solo sul training e poi trasformato sugli altri insiemi. La scelta dello scaling è obbligatoria per le reti neurali, in particolare per la stabilità della discesa del gradiente. I valori vengono convertiti in `float32` per compatibilità con PyTorch.

Imputer e scaler vengono salvati su disco per essere riutilizzati in fase di evaluation.

Le variabili target (tempo ed evento) vengono estratte come tuple di array numpy `float32`.

---

### Fase 4: Architettura della Rete Neurale e Addestramento

L’architettura scelta è un **MLP a 3 strati nascosti** con dimensioni `[64, 32, 16]` (più ampia rispetto alla versione precedente che aveva `[32, 16]`). Questa scelta aumenta leggermente la capacità, compensata da una regolarizzazione più forte:

- **Batch Normalization** dopo ogni strato lineare: stabilizza e accelera la convergenza.
- **Dropout** del 40% (`dropout=0.4`) dopo ogni strato (eccetto l’ultimo): riduce il co-adattamento dei neuroni.
- **Weight decay** (`weight_decay=0.01`) nell’ottimizzatore Adam: aggiunge una penalità L2 ai pesi, ulteriore forma di regolarizzazione.

L’ottimizzatore utilizza un learning rate di `0.001` (Adam), valore standard che garantisce una convergenza stabile.

L’addestramento viene eseguito per un massimo di **500 epoche** con **batch size = 64**. È attivata una callback di **Early Stopping** con pazienza di **25 epoche** sul loss di validation: se per 25 epoche consecutive il loss non migliora, l’addestramento si interrompe, restituendo i pesi della migliore epoca. Questo meccanismo previene l’overfitting e consente di sfruttare appieno le epoche utili.

Durante l’addestramento, il modello `CoxPH` di PyCox ottimizza la verosimiglianza parziale di Cox (loss `cox‑ph`). Al termine, la rete viene salvata in formato PyTorch (`.pt`).

---

### Fase 5: Valutazione Finale con C‑index Tempo‑dipendente

Dopo l’addestramento, viene calcolata la **baseline hazard** (`compute_baseline_hazards`) necessaria per ottenere le funzioni di sopravvivenza.

Per valutare la capacità discriminativa del modello, si utilizza il **concordance index tempo‑dipendente** (`concordance_td`) implementato in `EvalSurv`. Questo indice generalizza il C‑index tradizionale tenendo conto della natura dipendente dal tempo delle predizioni di sopravvivenza. Viene calcolato separatamente su training, validation e test.

I risultati vengono stampati a schermo per un rapido controllo di coerenza: il C‑index sul test dovrebbe essere prossimo a quello di validation, indicando che il modello generalizza bene.

---

**Riepilogo delle principali modifiche rispetto alla versione precedente**:

- **Missing indicators** per variabili con alta percentuale di missing.
- **Architettura leggermente più profonda** (3 strati: 32, 32, 16).
- **Separazione esplicita di un validation set** (15% del training originale) per l’early stopping.
- **Early stopping con pazienza aumentata a 25 epoche** per sfruttare al meglio le 500 epoche massime.
- **Weight decay** (`0.01`) come ulteriore regolarizzazione.
- **Batch size = 64** per una stima più stabile del gradiente.
- **Valutazione con C‑index tempo‑dipendente** (`concordance_td`) al posto del C‑index classico.
- **Salvataggio separato di imputer, scaler e lista feature** per garantire la piena riproducibilità in fase di evaluation.

L’esecuzione di questa pipeline produce un modello DeepSurv ottimizzato, accompagnato da tutti gli oggetti necessari per le successive fasi di validazione e interpretazione.

In [6]:
import os, warnings
import pandas as pd
import numpy as np
import joblib
import torchtuples as tt
from pycox.models import CoxPH
from pycox.evaluation import EvalSurv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
DATA_PATH     = "../data/dataset.csv"
MODEL_DIR     = "../models"
MODEL_PATH    = os.path.join(MODEL_DIR, "deepsurv_model.pt")
SCALER_PATH   = os.path.join(MODEL_DIR, "deepsurv_scaler.pkl")
IMPUTER_PATH  = os.path.join(MODEL_DIR, "deepsurv_imputer.pkl")
FEATURES_PATH = os.path.join(MODEL_DIR, "deepsurv_features.pkl")

os.makedirs(MODEL_DIR, exist_ok=True)

TIME_COL     = "follow_ups.days_to_follow_up"
EVENT_COL    = "demographic.vital_status_DEAD"
RANDOM_STATE = 26
TEST_SIZE    = 0.25
VAL_SIZE     = 0.15

NET_LAYERS   = [64, 32, 16]
NET_DROPOUT  = 0.4
NET_BATCHNORM = True

# ==============================================================================
# FASE 1: CARICAMENTO E SPLIT
# ==============================================================================
print("=" * 62)
print("  TRAINING PIPELINE - DEEPSURV (PyCox)")
print("=" * 62)

df_clean = pd.read_csv(DATA_PATH).replace(-1, np.nan)

X_raw = df_clean.drop(columns=[TIME_COL, EVENT_COL])

cols_with_many_nan = ['exposures.alcohol_days_per_week',
                      'exposures.alcohol_drinks_per_day',
                      'exposures.pack_years_smoked']

for col in cols_with_many_nan:
    if col in X_raw.columns:
        X_raw[f"{col}_is_missing"] = X_raw[col].isna().astype(float)

X_train_full, X_test, df_y_train_full, df_y_test = train_test_split(
    X_raw, df_clean[[TIME_COL, EVENT_COL]],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_clean[EVENT_COL]
)

# ==============================================================================
# FASE 2: FEATURE SELECTION (Frequenze) E VALIDATION SPLIT
# ==============================================================================
min_positives = 10
binary_cols   = [c for c in X_train_full.columns if X_train_full[c].nunique() == 2]
cols_to_drop  = [c for c in binary_cols if X_train_full[c].sum() < min_positives]

X_train_full = X_train_full.drop(columns=cols_to_drop)
X_test       = X_test.drop(columns=cols_to_drop, errors='ignore')
print(f"  [+] Rimosse {len(cols_to_drop)} feature troppo sparse.")

X_train, X_val, df_y_train, df_y_val = train_test_split(
    X_train_full, df_y_train_full,
    test_size=VAL_SIZE, random_state=RANDOM_STATE, stratify=df_y_train_full[EVENT_COL]
)

joblib.dump(X_train.columns.tolist(), FEATURES_PATH)

# ==============================================================================
# FASE 3: IMPUTAZIONE E SCALING COMPLETO (Senza Leakage)
# ==============================================================================
imputer = IterativeImputer(random_state=RANDOM_STATE, max_iter=10)
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_val_imp   = pd.DataFrame(imputer.transform(X_val),       columns=X_val.columns)
X_test_imp  = pd.DataFrame(imputer.transform(X_test),      columns=X_test.columns)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp).astype('float32')
X_val_scaled   = scaler.transform(X_val_imp).astype('float32')
X_test_scaled  = scaler.transform(X_test_imp).astype('float32')

joblib.dump(imputer, IMPUTER_PATH)
joblib.dump(scaler,  SCALER_PATH)

get_target = lambda df: (df[TIME_COL].values.astype('float32'), df[EVENT_COL].values.astype('float32'))
y_train = get_target(df_y_train)
y_val   = get_target(df_y_val)
y_test  = get_target(df_y_test)

# ==============================================================================
# FASE 4: ARCHITETTURA RETE E TRAINING
# ==============================================================================
in_features = X_train_scaled.shape[1]

net = tt.practical.MLPVanilla(
    in_features, NET_LAYERS, 1,
    batch_norm=NET_BATCHNORM,
    dropout=NET_DROPOUT
)

optimizer = tt.optim.Adam(lr=0.001, weight_decay=0.01)
model = CoxPH(net, optimizer)

callbacks = [tt.callbacks.EarlyStopping(patience=25)]

print(f"  [+] Avvio addestramento su {in_features} feature...")
log = model.fit(X_train_scaled, y_train, batch_size=64, epochs=500,
                callbacks=callbacks, val_data=(X_val_scaled, y_val), verbose=False)

model.save_net(MODEL_PATH)

# ==============================================================================
# FASE 5: VALUTAZIONE FINALE C-INDEX
# ==============================================================================
_ = model.compute_baseline_hazards()

def evaluate_c_index(model_obj, x_data, y_data):
    durations, events = y_data
    surv_df = model_obj.predict_surv_df(x_data)
    ev = EvalSurv(surv_df, durations, events, censor_surv='km')
    return ev.concordance_td()

c_index_train = evaluate_c_index(model, X_train_scaled, y_train)
c_index_val   = evaluate_c_index(model, X_val_scaled,   y_val)
c_index_test  = evaluate_c_index(model, X_test_scaled,  y_test)

print("=" * 62)
print("  RISULTATI DEEPSURV")
print("=" * 62)
print(f"  Epoche ottimali : {len(log.to_pandas())}")
print(f"  C-index TRAIN   : {c_index_train:.4f}")
print(f"  C-index VAL     : {c_index_val:.4f}")
print(f"  C-index TEST    : {c_index_test:.4f}")
print("=" * 62)

  TRAINING PIPELINE - DEEPSURV (PyCox)
  [+] Rimosse 1 feature troppo sparse.
  [+] Avvio addestramento su 35 feature...
  RISULTATI DEEPSURV
  Epoche ottimali : 95
  C-index TRAIN   : 0.8173
  C-index VAL     : 0.6079
  C-index TEST    : 0.7078


# DeepSurv (PyCox) – Evaluation & Explainable AI (Versione Aggiornata)

Questo notebook valuta le performance della rete neurale **DeepSurv** addestrata con la pipeline aggiornata.
Rispetto alla versione precedente, l’analisi è stata potenziata con una **valutazione multi‑run** (5 esecuzioni indipendenti) per ottenere stime robuste di metriche (C‑index, IBS, AUC tempo‑dipendente) e con una procedura di explainability che opera direttamente sui dati **imputati ma non scalati**, rendendo i grafici SHAP clinicamente interpretabili (valori originali delle feature).

---

### Fase 1: Ricostruzione dell’Ambiente e Preprocessing (Identico al Training)

Per garantire che la valutazione avvenga nelle stesse condizioni del training, l’intera pipeline di preprocessing viene riprodotta fedelmente:

1. **Caricamento e pulizia** – Il dataset viene letto e i placeholder `-1` convertiti in `NaN`.
2. **Missing indicators** – Per le variabili con alta percentuale di missing (`exposures.alcohol_days_per_week`, `exposures.alcohol_drinks_per_day`, `exposures.pack_years_smoked`) vengono create le stesse colonne binarie `_is_missing` utilizzate in training.
3. **Suddivisione train/test** – Utilizzando lo stesso `random_state` (26) e la stessa frazione di test (0.25), si isolano gli stessi insiemi usati durante l’addestramento. La stratificazione sull’evento garantisce che la proporzione di censure sia identica.
4. **Feature selection** – Viene ricalcolata la lista delle feature binarie con meno di 10 positivi (soglia `min_positives`) **sul solo training set**, esattamente come in fase di training. Le colonne eliminate vengono rimosse anche dal test.
5. **Allineamento delle feature** – Viene caricato l’elenco delle feature finali salvato in `deepsurv_features.pkl`. Se per qualche motivo alcune colonne mancassero (ad esempio perché durante il training alcune erano state eliminate e poi ricreate), vengono aggiunte con valore zero e riordinate. Questo passaggio garantisce che la struttura dei dati di test sia perfettamente allineata a quella attesa dal modello.
6. **Imputazione MICE e scaling** – L’imputer e lo scaler salvati durante il training vengono caricati e applicati **solo con `transform`** (nessun riadattamento) ai dati di train e test, prevenendo qualsiasi forma di *data leakage*.

Infine, vengono preparati gli oggetti necessari per le metriche:
- `y_train_full_sksurv` e `y_test_sksurv` in formato `Surv` per le funzioni di `scikit-survival`.
- Le tuple `(tempo, evento)` per PyCox.
- I tempi per l’IBS (20 punti tra il 10° e il 90° percentile dei tempi di test) e gli orizzonti temporali per l’AUC (1, 3, 5 anni) – vengono esclusi quelli oltre il follow‑up massimo.

---

### Fase 2: Valutazione Multi‑Run (5 Esecuzioni Indipendenti)

Le reti neurali sono sensibili alla suddivisione tra training e validation e all’inizializzazione casuale dei pesi. Per ottenere stime affidabili delle performance, vengono eseguite **5 run indipendenti**:

- Per ogni run `i` si imposta un seed diverso (`RANDOM_STATE + i`) per:
  - la suddivisione **training/validation** (15% del training originale riservato alla validation);
  - l’inizializzazione dei pesi della rete (`torch.manual_seed`, `np.random.seed`).
- L’imputazione e lo scaling vengono riapplicati **dopo** lo split, sempre utilizzando gli stessi oggetti `imputer` e `scaler` (già addestrati sull’intero training set), in modo che le statistiche rimangano fisse.
- Il modello viene addestrato con la stessa architettura, ottimizzatore e early stopping (pazienza 25) descritti nella pipeline di training.
- Al termine di ogni run si calcolano:
  - **C‑index** sul test set (tramite `concordance_index_censored` sui risk scores);
  - **Integrated Brier Score (IBS)** previa interpolazione delle curve di sopravvivenza sulla griglia temporale comune;
  - **AUC tempo‑dipendente** per ciascun orizzonte valido (1, 3, 5 anni) usando `cumulative_dynamic_auc`.

Alla fine delle 5 run si ottengono per ogni metrica **media e deviazione standard**, fornendo una visione realistica della variabilità del modello.

---

### Fase 3: Aggregazione dei Risultati e Selezione del Modello Rappresentativo

Vengono stampate le metriche aggregate:

- C‑index medio ± std
- IBS medio ± std
- Per ogni orizzonte AUC medio ± std

Queste statistiche offrono una valutazione robusta e quantificano l’incertezza legata alla casualità dell’addestramento.

Per le successive analisi di explainability (Permutation Importance e SHAP) è necessario un modello **rappresentativo**. Viene selezionato il run il cui C‑index è più vicino alla media di tutti i run. In questo modo le visualizzazioni XAI rifletteranno il comportamento tipico del modello.

---

### Fase 4: Stratificazione del Rischio (Validazione Clinica)

Utilizzando il modello rappresentativo e i risk scores da esso prodotti sul test set, i pazienti vengono divisi in due coorti in base alla **mediana** del rischio:

- **Alto Rischio**: risk score > mediana
- **Basso Rischio**: risk score ≤ mediana

Le curve di sopravvivenza medie per ciascun gruppo si ottengono semplicemente mediando le funzioni di sopravvivenza predette (`surv_df`) per le colonne corrispondenti. Il grafico a gradini risultante mostra visivamente la capacità del modello di separare le popolazioni con prognosi diversa, un aspetto fondamentale per la validazione clinica.

---

### Fase 5: Explainability Globale – Permutation Feature Importance

Per comprendere quali variabili guidano maggiormente le predizioni a livello di popolazione, si applica la **Permutation Feature Importance** sul modello rappresentativo.

- Poiché la funzione `permutation_importance` di scikit‑learn richiede un estimatore con metodo `predict`, si definisce una funzione `deepsurv_scorer` che accetta i dati **scalati** (come usati dal modello) e restituisce il C‑index.
- La permutation viene eseguita con 10 ripetizioni, mescolando i valori di ogni feature alla volta e misurando la riduzione del C‑index.
- Le importanze (riduzione media) vengono convertite in **percentuali relative** e visualizzate in un grafico a barre orizzontali con le barre di errore (deviazione standard sulle ripetizioni).
Questo metodo è indipendente dal tipo di modello e fornisce una visione globale dell’impatto di ogni variabile sulle prestazioni.

---

### Fase 6: Explainability Locale – SHAP su Dati Imputati (Valori Originali)

Le reti neurali sono spesso considerate “scatole nere”. SHAP permette di aprire questa scatola, ma con un accorgimento cruciale: per ottenere grafici leggibili clinicamente, i valori SHAP devono essere associati ai **valori originali delle feature** (es. età = 65 anni), non ai valori standardizzati (es. età = 1.24).

A questo scopo si definisce una funzione `predict_risk_shap` che:

- accetta in input un dataframe **imputato** (con valori nello spazio originale);
- applica internamente lo `scaler` (trasformando i dati nel formato atteso dal modello);
- restituisce i risk scores.

L’`Explainer` SHAP viene inizializzato con questa funzione e un **background dataset** di 100 campioni estratti casualmente dal training set imputato (con seed fisso).
Vengono generati:

- **SHAP Summary Plot (beeswarm)** – mostra la distribuzione dei valori SHAP per ogni feature sull’intero test set, con il colore che indica il valore della feature (rosso = alto, blu = basso). Le feature includono anche le missing indicators, permettendo di capire se la mancanza stessa di un dato influenza il rischio.
- **SHAP Waterfall Plot** – per i due pazienti con il rischio **più alto** e **più basso** nel test set. Questi grafici partono dal valore base (rischio medio atteso sulla popolazione di background) e mostrano il contributo additivo di ogni feature, con i relativi valori originali riportati sull’asse y. In questo modo un clinico può immediatamente comprendere perché un paziente ha un rischio elevato (es. “elevato T_stage”, “età avanzata”, “presenza di metastasi”).

Tutti i grafici vengono salvati nella cartella `results/` per una successiva inclusione in report o presentazioni.

---

**Riepilogo delle principali novità rispetto alla versione precedente**:

- **Valutazione multi‑run (5 esecuzioni)** con stima di media e deviazione standard per tutte le metriche.
- **Selezione del modello rappresentativo** per le analisi XAI.
- **SHAP su dati imputati** con wrapper che applica lo scaling internamente, garantendo interpretabilità clinica (valori originali delle feature).
- **Permutation Feature Importance** calcolata sul modello rappresentativo.
- **Gestione esplicita delle feature mancanti** tramite indicatori, inclusi nei plot SHAP.

L’esecuzione di questo notebook fornisce un quadro completo delle performance e delle logiche del modello DeepSurv, soddisfacendo sia le esigenze di validazione quantitativa che quelle di interpretabilità clinica.

In [1]:
import os, warnings
import pandas as pd
import numpy as np
import joblib
import torchtuples as tt
import shap
import matplotlib.pyplot as plt
from pycox.models import CoxPH
from sksurv.util import Surv
from sksurv.metrics import (concordance_index_censored,
                            integrated_brier_score,
                            cumulative_dynamic_auc)
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURAZIONE E PATHS
# ==============================================================================
RESULTS_DIR = "results"
DATA_PATH = "../data/dataset.csv"
MODEL_DIR = "../models"
MODEL_PATH = os.path.join(MODEL_DIR, "deepsurv_model.pt")
SCALER_PATH = os.path.join(MODEL_DIR, "deepsurv_scaler.pkl")
IMPUTER_PATH = os.path.join(MODEL_DIR, "deepsurv_imputer.pkl")
FEATURES_PATH = os.path.join(MODEL_DIR, "deepsurv_features.pkl")

os.makedirs(RESULTS_DIR, exist_ok=True)

for _f in [MODEL_PATH, SCALER_PATH, IMPUTER_PATH, FEATURES_PATH]:
    if not os.path.exists(_f):
        raise FileNotFoundError(f"File {_f} non trovato. Esegui prima il training.")

TIME_COL = "follow_ups.days_to_follow_up"
EVENT_COL = "demographic.vital_status_DEAD"
RANDOM_STATE = 26
TEST_SIZE = 0.25
VAL_SIZE = 0.15
N_RUNS = 5

NET_LAYERS = [64, 32, 16]
NET_DROPOUT = 0.4
NET_BATCHNORM = True

# ==============================================================================
# FASE 1: RICOSTRUZIONE AMBIENTE E PREPROCESSING (UNA SOLA VOLTA)
# ==============================================================================
print("=" * 62)
print("  EVALUATION & XAI - DEEPSURV  (media su 5 run)")
print("=" * 62)

df_clean = pd.read_csv(DATA_PATH).replace(-1, np.nan)

X_raw = df_clean.drop(columns=[TIME_COL, EVENT_COL])

cols_with_many_nan = ['exposures.alcohol_days_per_week',
                      'exposures.alcohol_drinks_per_day',
                      'exposures.pack_years_smoked']

for col in cols_with_many_nan:
    if col in X_raw.columns:
        X_raw[f"{col}_is_missing"] = X_raw[col].isna().astype(float)

X_train_full, X_test, df_y_train_full, df_y_test = train_test_split(
    X_raw, df_clean[[TIME_COL, EVENT_COL]],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df_clean[EVENT_COL]
)

min_positives = 10
binary_cols = [c for c in X_train_full.columns if X_train_full[c].nunique() == 2]
cols_to_drop = [c for c in binary_cols if X_train_full[c].sum() < min_positives]

X_train_full = X_train_full.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop, errors='ignore')

feature_cols_final = joblib.load(FEATURES_PATH)

for col in feature_cols_final:
    if col not in X_train_full.columns:
        X_train_full[col] = 0
    if col not in X_test.columns:
        X_test[col] = 0

X_train_full = X_train_full[feature_cols_final]
X_test = X_test[feature_cols_final]

y_train_full_sksurv = Surv.from_arrays(
    event=df_y_train_full[EVENT_COL].astype(bool),
    time=df_y_train_full[TIME_COL]
)
y_test_sksurv = Surv.from_arrays(
    event=df_y_test[EVENT_COL].astype(bool),
    time=df_y_test[TIME_COL]
)

imputer = joblib.load(IMPUTER_PATH)
scaler = joblib.load(SCALER_PATH)

X_train_full_imp = pd.DataFrame(imputer.transform(X_train_full), columns=X_train_full.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

X_train_full_scaled = scaler.transform(X_train_full_imp).astype('float32')
X_test_scaled = scaler.transform(X_test_imp).astype('float32')

y_train_full_pycox = (
    df_y_train_full[TIME_COL].values.astype('float32'),
    df_y_train_full[EVENT_COL].values.astype('float32')
)

in_features = X_train_full_scaled.shape[1]
print(f"  [+] Ambiente ricostruito: {in_features} feature attive.")

lb = np.percentile(df_y_test[TIME_COL].values, 10)
ub = np.percentile(df_y_test[TIME_COL].values, 90)
ibs_times = np.linspace(lb, ub, 20)

time_max_safe = min(df_y_train_full[TIME_COL].max(), df_y_test[TIME_COL].max())
horizon_labels = {1: 365, 3: 1095, 5: 1825}
valid_horizons = {lbl: days for lbl, days in horizon_labels.items() if days < time_max_safe}
skipped = [lbl for lbl, days in horizon_labels.items() if days >= time_max_safe]

# ==============================================================================
# FASE 2: LOOP SU N_RUNS — TRAINING E METRICHE
# ==============================================================================
print(f"\n  [+] Avvio {N_RUNS} run indipendenti...")
print("=" * 62)

runs_c_index = []
runs_ibs = []
runs_auc = {lbl: [] for lbl in valid_horizons}
runs_models = []
runs_risk = []
runs_surv = []

for run in range(N_RUNS):
    run_seed = RANDOM_STATE + run

    X_tr, X_vl, df_ytr, df_yvl = train_test_split(
        X_train_full, df_y_train_full,
        test_size=VAL_SIZE, random_state=run_seed,
        stratify=df_y_train_full[EVENT_COL]
    )

    X_tr_imp = pd.DataFrame(imputer.transform(X_tr), columns=X_tr.columns)
    X_vl_imp = pd.DataFrame(imputer.transform(X_vl), columns=X_vl.columns)

    X_tr_scaled = scaler.transform(X_tr_imp).astype('float32')
    X_vl_scaled = scaler.transform(X_vl_imp).astype('float32')

    y_tr = (df_ytr[TIME_COL].values.astype('float32'), df_ytr[EVENT_COL].values.astype('float32'))
    y_vl = (df_yvl[TIME_COL].values.astype('float32'), df_yvl[EVENT_COL].values.astype('float32'))

    import torch;

    torch.manual_seed(run_seed);
    np.random.seed(run_seed)
    net = tt.practical.MLPVanilla(in_features, NET_LAYERS, 1, NET_BATCHNORM, NET_DROPOUT)
    model = CoxPH(net, tt.optim.Adam(lr=0.001, weight_decay=0.01))

    model.fit(X_tr_scaled, y_tr, batch_size=64, epochs=500,
              callbacks=[tt.callbacks.EarlyStopping(patience=25)],
              val_data=(X_vl_scaled, y_vl), verbose=False)

    _ = model.compute_baseline_hazards(X_train_full_scaled, y_train_full_pycox)

    risk_test = model.predict(X_test_scaled).flatten()
    surv_df = model.predict_surv_df(X_test_scaled)

    c_idx = concordance_index_censored(
        y_test_sksurv['event'], y_test_sksurv['time'], risk_test
    )[0]

    preds_interp = np.array([
        np.interp(ibs_times, surv_df.index.values, surv_df.iloc[:, i].values)
        for i in range(surv_df.shape[1])
    ])
    ibs_val = integrated_brier_score(
        y_train_full_sksurv, y_test_sksurv, preds_interp, ibs_times
    )

    auc_vals = {}
    if valid_horizons:
        auc_times = np.array(list(valid_horizons.values()), dtype=float)
        auc_result, _ = cumulative_dynamic_auc(
            y_train_full_sksurv, y_test_sksurv, risk_test, auc_times
        )
        for lbl, av in zip(valid_horizons.keys(), auc_result):
            auc_vals[lbl] = av

    runs_c_index.append(c_idx)
    runs_ibs.append(ibs_val)
    for lbl in valid_horizons:
        runs_auc[lbl].append(auc_vals[lbl])
    runs_models.append(model)
    runs_risk.append(risk_test)
    runs_surv.append(surv_df)

    print(f"  Run {run + 1}/{N_RUNS} — C-index: {c_idx:.4f} | IBS: {ibs_val:.4f}")

# ==============================================================================
# FASE 3: AGGREGAZIONE RISULTATI
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 3: METRICHE AGGREGATE (media ± std)")
print("=" * 62)

mean_c = np.mean(runs_c_index)
std_c = np.std(runs_c_index)
mean_ibs = np.mean(runs_ibs)
std_ibs = np.std(runs_ibs)

print(f"  C-index : {mean_c:.4f} ± {std_c:.4f}")
print(f"  IBS     : {mean_ibs:.4f} ± {std_ibs:.4f}")

if skipped:
    print(f"  [!] Orizzonti esclusi (oltre il follow-up massimo): {skipped} anni")

mean_auc_overall = []
for lbl, days in valid_horizons.items():
    m = np.mean(runs_auc[lbl])
    s = np.std(runs_auc[lbl])
    mean_auc_overall.append(m)
    print(f"  AUC a {lbl} anno/i ({days} giorni) : {m:.4f} ± {s:.4f}")

if mean_auc_overall:
    print(f"  AUC media orizzonti validi         : {np.mean(mean_auc_overall):.4f}")

best_run_idx = int(np.argmin(np.abs(np.array(runs_c_index) - mean_c)))
model = runs_models[best_run_idx]
risk_test = runs_risk[best_run_idx]
surv_df = runs_surv[best_run_idx]
print(f"\n  [+] Modello rappresentativo: run {best_run_idx + 1} (C-index: {runs_c_index[best_run_idx]:.4f})")

# ==============================================================================
# FASE 4: STRATIFICAZIONE DEL RISCHIO
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 4: STRATIFICAZIONE CLINICA")
print("=" * 62)

median_risk = np.median(risk_test)
mask_high = risk_test > median_risk
mask_low = risk_test <= median_risk

plt.figure(figsize=(10, 6))
plt.step(surv_df.index, surv_df.loc[:, mask_low].mean(axis=1), where="post", label="Basso Rischio", color="#2ca02c")
plt.step(surv_df.index, surv_df.loc[:, mask_high].mean(axis=1), where="post", label="Alto Rischio", color="#d62728")
plt.title("Sopravvivenza Media: Alto vs Basso Rischio (DeepSurv)")
plt.ylabel("S(t)")
plt.xlabel("Giorni")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_survival_curves.png"), dpi=150)
plt.close()
print(f"  [+] Grafico curve salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 5: XAI GLOBALE (PERMUTATION FEATURE IMPORTANCE)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 5: PERMUTATION FEATURE IMPORTANCE")
print("=" * 62)
print("  Calcolo in corso (10 ripetizioni)...")


def deepsurv_scorer(est, X_in, y_in):
    risk = est.predict(X_in.astype('float32')).flatten()
    return concordance_index_censored(y_in['event'], y_in['time'], risk)[0]


perm = permutation_importance(
    model, X_test_scaled, y_test_sksurv,
    scoring=deepsurv_scorer,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

imp_mean = np.maximum(perm.importances_mean, 0)
imp_pct = 100.0 * imp_mean / (imp_mean.sum() + 1e-12)

df_imp = pd.DataFrame({
    "Feature": X_test.columns,
    "Importanza (%)": imp_pct,
    "Std": perm.importances_std,
}).sort_values("Importanza (%)", ascending=False)

df_imp_top = df_imp.head(15)

fig2, ax2 = plt.subplots(figsize=(9, 7))
ax2.barh(df_imp_top["Feature"][::-1], df_imp_top["Importanza (%)"][::-1],
         color="#1f77b4", xerr=df_imp_top["Std"][::-1], capsize=3)
ax2.set_xlabel("Importanza relativa (%)", fontsize=11)
ax2.set_title("Permutation Feature Importance – DeepSurv", fontsize=12, fontweight="bold")
ax2.grid(axis="x", alpha=0.35, linestyle="--")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_permutation_importance.png"), dpi=150)
plt.close()
print(f"  [+] Grafico Permutation salvato in: {RESULTS_DIR}")

# ==============================================================================
# FASE 6: XAI LOCALE (SHAP)
# ==============================================================================
print("\n" + "=" * 62)
print("  FASE 6: XAI LOCALE (SHAP)")
print("=" * 62)


def predict_risk_shap(X_imp):
    X_scaled = scaler.transform(X_imp).astype('float32')
    return model.predict(X_scaled).flatten()


background = shap.sample(X_train_full_imp, 100, random_state=RANDOM_STATE)
explainer = shap.Explainer(predict_risk_shap, background)

print("  Calcolo SHAP (BeeSwarm)...")
shap_values = explainer(X_test_imp)

plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP Summary: Impatto sul Rischio (con Missing Indicators)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "deepsurv_shap_summary.png"), dpi=150, bbox_inches="tight")
plt.close()
print("  [+] SHAP Summary plot salvato")

idx_alto = int(np.argmax(risk_test))
idx_basso = int(np.argmin(risk_test))

for label, idx in [("ALTO RISCHIO", idx_alto), ("BASSO RISCHIO", idx_basso)]:
    plt.figure(figsize=(14, 7))
    try:
        shap.plots.waterfall(shap_values[idx], max_display=13, show=False, matplotlib=True)
    except TypeError:
        shap.plots.waterfall(shap_values[idx], max_display=13, show=False)

    ax_wf = plt.gca()
    plt.title(f"SHAP Waterfall – DeepSurv – Paziente {label}", fontsize=12, fontweight="bold", pad=12)
    ax_wf.tick_params(axis="y", labelsize=9)
    for text in ax_wf.texts:
        if "f(x)" not in text.get_text() and "base value" not in text.get_text():
            text.set_fontsize(8)
    plt.subplots_adjust(left=0.42, right=0.95, top=0.90, bottom=0.10)
    ax_wf.grid(axis="x", alpha=0.3, linestyle="--")

    fname = f"deepsurv_shap_waterfall_{label.lower().replace(' ', '_')}.png"
    plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=150, bbox_inches="tight")
    plt.close()

print("  [+] SHAP Waterfall plots salvati per pazienti ad alto e basso rischio")

# ==============================================================================
# RIEPILOGO FINALE
# ==============================================================================
print("\n" + "=" * 62)
print("  RIEPILOGO FINALE – DeepSurv")
print("=" * 62)
print(f"  Run eseguiti    : {N_RUNS}")
print(f"  C-index         : {mean_c:.4f} ± {std_c:.4f}")
print(f"  IBS             : {mean_ibs:.4f} ± {std_ibs:.4f}")
for lbl, days in valid_horizons.items():
    m = np.mean(runs_auc[lbl])
    s = np.std(runs_auc[lbl])
    print(f"  AUC {lbl}yr        : {m:.4f} ± {s:.4f}")
print(f"  Grafici salvati in: {RESULTS_DIR}")
print("=" * 62)

  EVALUATION & XAI - DEEPSURV  (media su 5 run)
  [+] Ambiente ricostruito: 35 feature attive.

  [+] Avvio 5 run indipendenti...
  Run 1/5 — C-index: 0.7226 | IBS: 0.1850
  Run 2/5 — C-index: 0.7157 | IBS: 0.1908
  Run 3/5 — C-index: 0.7114 | IBS: 0.1945
  Run 4/5 — C-index: 0.6825 | IBS: 0.2027
  Run 5/5 — C-index: 0.6959 | IBS: 0.2083

  FASE 3: METRICHE AGGREGATE (media ± std)
  C-index : 0.7056 ± 0.0145
  IBS     : 0.1963 ± 0.0083
  AUC a 1 anno/i (365 giorni) : 0.7477 ± 0.0238
  AUC a 3 anno/i (1095 giorni) : 0.7347 ± 0.0192
  AUC a 5 anno/i (1825 giorni) : 0.6221 ± 0.0349
  AUC media orizzonti validi         : 0.7015

  [+] Modello rappresentativo: run 3 (C-index: 0.7114)

  FASE 4: STRATIFICAZIONE CLINICA
  [+] Grafico curve salvato in: results

  FASE 5: PERMUTATION FEATURE IMPORTANCE
  Calcolo in corso (10 ripetizioni)...
  [+] Grafico Permutation salvato in: results

  FASE 6: XAI LOCALE (SHAP)
  Calcolo SHAP (BeeSwarm)...
  [+] SHAP Summary plot salvato
  [+] SHAP Waterfall